# PAATRA Step 2: Kill Gate — Vocabulary Subsetting

**The question:** Can you chop a model's vocabulary from 152K → 30K/20K/10K tokens and still get coherent output?

If yes → the linguistic saturation hypothesis holds, and PAATRA is viable.  
If the model produces garbage even at 30K → the approach may be fundamentally limited.

**Model:** Qwen2.5-0.5B (ungated, 151,936 vocab, 28% embedding overhead)  
**Method:** Count token frequencies on English text, keep top-K, zero out the rest, run inference.  
**No training involved** — pure subsetting + inference.

In [ ]:
!pip install -q torch transformers accelerate datasets

## 1. Load Model & Tokenizer

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from collections import Counter
import copy
import gc
import time

MODEL_ID = "Qwen/Qwen2.5-0.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map=DEVICE,
)
model.eval()

print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Model loaded on {DEVICE}")

## 2. Baseline — Generate with Full Vocabulary

First, let's see what the model produces with its full 152K vocabulary.

In [ ]:
TEST_PROMPTS = [
    "The process of photosynthesis involves",
    "Water boils at 100 degrees Celsius because",
    "The capital of France is",
    "To solve 24 divided by 6, we",
    "The three states of matter are",
]

def generate_responses(model, tokenizer, prompts, max_new_tokens=60, label=""):
    """Generate from a list of prompts and print results."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    responses = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
            )
        text = tokenizer.decode(output[0], skip_special_tokens=True)
        response = text[len(prompt):].strip()
        responses.append(response)
        print(f"\n  Prompt: {prompt}")
        print(f"  Output: {response[:200]}")
    return responses

baseline_responses = generate_responses(model, tokenizer, TEST_PROMPTS, label="BASELINE (full 152K vocab)")

## 3. Count Token Frequencies on English Text

Use a sample of English text to find which tokens the model actually uses most often.

In [ ]:
print("Loading English corpus (wikitext-103)...")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

# Sample a manageable chunk
NUM_SAMPLES = 50_000
texts = [t for t in dataset["text"][:NUM_SAMPLES] if len(t.strip()) > 20]
print(f"Using {len(texts):,} non-empty text samples")

# Count token frequencies
print("Counting token frequencies...")
token_counts = Counter()
for text in texts:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    token_counts.update(token_ids)

total_tokens = sum(token_counts.values())
unique_tokens = len(token_counts)
print(f"Total tokens: {total_tokens:,}")
print(f"Unique tokens seen: {unique_tokens:,} out of {tokenizer.vocab_size:,} ({unique_tokens/tokenizer.vocab_size*100:.1f}%)")

# How many tokens cover 99% of usage?
sorted_counts = sorted(token_counts.values(), reverse=True)
cumsum = 0
for i, count in enumerate(sorted_counts):
    cumsum += count
    if cumsum >= total_tokens * 0.99:
        print(f"Top {i+1:,} tokens cover 99% of all token usage")
        break

## 4. Build Subset Vocabularies

For each K, keep the top-K most frequent tokens + all special tokens.  
Zero out embedding rows for tokens not in the subset.

In [ ]:
def get_special_token_ids(tokenizer):
    """Collect all special token IDs that must always be kept."""
    special_ids = set()
    if tokenizer.bos_token_id is not None:
        special_ids.add(tokenizer.bos_token_id)
    if tokenizer.eos_token_id is not None:
        special_ids.add(tokenizer.eos_token_id)
    if tokenizer.pad_token_id is not None:
        special_ids.add(tokenizer.pad_token_id)
    if tokenizer.unk_token_id is not None:
        special_ids.add(tokenizer.unk_token_id)
    if hasattr(tokenizer, 'additional_special_tokens_ids'):
        special_ids.update(tokenizer.additional_special_tokens_ids)
    # Keep byte-level fallback tokens (usually IDs 0-255 or similar)
    # These let the model handle any unknown byte sequence
    return special_ids

special_ids = get_special_token_ids(tokenizer)
print(f"Special tokens to always keep: {len(special_ids)}")

def build_keep_set(token_counts, K, special_ids):
    """Return the set of token IDs to keep: top-K by frequency + specials."""
    top_k_ids = {tid for tid, _ in token_counts.most_common(K)}
    keep = top_k_ids | special_ids
    return keep

# Preview
for K in [5_000, 10_000, 20_000, 30_000, 50_000]:
    keep = build_keep_set(token_counts, K, special_ids)
    # Check coverage: what % of corpus tokens are in the keep set?
    covered = sum(c for tid, c in token_counts.items() if tid in keep)
    print(f"  K={K:>6,}: keeping {len(keep):>6,} tokens, corpus coverage = {covered/total_tokens*100:.2f}%")

## 5. The Kill Gate Experiment

For each K, zero out embeddings of tokens not in the keep set, then generate.  
We modify the embedding matrix in-place and restore it after each test.

In [ ]:
# Save original embeddings
embed_layer = model.get_input_embeddings()
original_weight = embed_layer.weight.data.clone()

# If model has tied output embeddings (lm_head), we handle that too
has_lm_head = hasattr(model, 'lm_head') and model.lm_head.weight.data_ptr() != embed_layer.weight.data_ptr()
if has_lm_head:
    original_lm_head = model.lm_head.weight.data.clone()

print(f"Original embedding shape: {original_weight.shape}")
print(f"LM head separate: {has_lm_head}")

In [ ]:
VOCAB_SIZES = [50_000, 30_000, 20_000, 10_000, 5_000]

all_results = {}

for K in VOCAB_SIZES:
    keep_set = build_keep_set(token_counts, K, special_ids)

    # Zero out embeddings for tokens NOT in keep set
    mask = torch.ones(original_weight.shape[0], dtype=torch.bool)
    for tid in keep_set:
        if tid < mask.shape[0]:
            mask[tid] = False  # False = don't zero out

    # Apply: zero out rows for pruned tokens
    embed_layer.weight.data.copy_(original_weight)
    embed_layer.weight.data[mask] = 0.0

    # Also zero out lm_head rows if separate
    if has_lm_head:
        model.lm_head.weight.data.copy_(original_lm_head)
        model.lm_head.weight.data[mask] = 0.0

    zeroed_count = mask.sum().item()
    kept_count = (~mask).sum().item()

    # Calculate parameter savings
    hidden_dim = original_weight.shape[1]
    saved_params = zeroed_count * hidden_dim
    total_params = sum(p.numel() for p in model.parameters())

    label = f"K={K:,} ({kept_count:,} tokens kept, {zeroed_count:,} zeroed, {saved_params/1e6:.1f}M params freed)"
    responses = generate_responses(model, tokenizer, TEST_PROMPTS, label=label)
    all_results[K] = responses

# Restore original weights
embed_layer.weight.data.copy_(original_weight)
if has_lm_head:
    model.lm_head.weight.data.copy_(original_lm_head)

print("\nOriginal weights restored.")

## 6. Side-by-Side Comparison

In [ ]:
print(f"\n{'='*80}")
print(f"  KILL GATE RESULTS — Side by Side")
print(f"{'='*80}")

for i, prompt in enumerate(TEST_PROMPTS):
    print(f"\n{'─'*80}")
    print(f"  PROMPT: {prompt}")
    print(f"{'─'*80}")
    print(f"  {'Baseline (152K)':15s}: {baseline_responses[i][:150]}")
    for K in VOCAB_SIZES:
        label = f"K={K//1000}K"
        print(f"  {label:15s}: {all_results[K][i][:150]}")

print(f"\n{'='*80}")
print("VERDICT:")
print("  - If outputs are coherent at 30K and 20K → linguistic saturation confirmed")
print("  - If outputs degrade at 10K but hold at 20K → saturation point is ~15-20K")
print("  - If outputs are garbage even at 50K → approach needs rethinking")
print(f"{'='*80}")

## 7. Quantitative Check — Perplexity on Held-Out Text

Beyond eyeballing outputs, measure perplexity at each vocab size.  
Lower is better. We expect a gradual increase as K shrinks, with a sharp knee at the saturation point.

In [ ]:
def compute_perplexity(model, tokenizer, texts, max_length=512):
    """Compute perplexity on a list of texts."""
    total_loss = 0.0
    total_tokens = 0

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
        if inputs["input_ids"].shape[1] < 2:
            continue
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        total_loss += outputs.loss.item() * (inputs["input_ids"].shape[1] - 1)
        total_tokens += inputs["input_ids"].shape[1] - 1

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    return torch.exp(torch.tensor(avg_loss)).item()

# Use held-out wikitext for perplexity
print("Loading held-out test set...")
test_data = load_dataset("wikitext", "wikitext-103-raw-v1", split="test")
test_texts = [t for t in test_data["text"][:2000] if len(t.strip()) > 50][:200]
print(f"Using {len(test_texts)} test texts")

# Baseline perplexity
print("\nComputing baseline perplexity...")
baseline_ppl = compute_perplexity(model, tokenizer, test_texts)
print(f"  Baseline (152K vocab): PPL = {baseline_ppl:.2f}")

# Perplexity at each K
ppl_results = {"baseline": baseline_ppl}

for K in VOCAB_SIZES:
    keep_set = build_keep_set(token_counts, K, special_ids)

    mask = torch.ones(original_weight.shape[0], dtype=torch.bool)
    for tid in keep_set:
        if tid < mask.shape[0]:
            mask[tid] = False

    embed_layer.weight.data.copy_(original_weight)
    embed_layer.weight.data[mask] = 0.0
    if has_lm_head:
        model.lm_head.weight.data.copy_(original_lm_head)
        model.lm_head.weight.data[mask] = 0.0

    ppl = compute_perplexity(model, tokenizer, test_texts)
    ppl_results[K] = ppl
    print(f"  K={K:>6,}: PPL = {ppl:.2f}  (vs baseline {baseline_ppl:.2f}, ratio = {ppl/baseline_ppl:.2f}x)")

# Restore
embed_layer.weight.data.copy_(original_weight)
if has_lm_head:
    model.lm_head.weight.data.copy_(original_lm_head)

print("\nWeights restored.")

## 8. Plot the Saturation Curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot vocab size vs perplexity
k_values = sorted([k for k in ppl_results if k != "baseline"])
ppls = [ppl_results[k] for k in k_values]

ax.plot(k_values, ppls, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Subsetted model')
ax.axhline(y=baseline_ppl, color='#2ecc71', linestyle='--', linewidth=2, label=f'Baseline (full 152K): PPL={baseline_ppl:.1f}')

# Mark the "saturation zone"
ax.axvspan(10_000, 30_000, alpha=0.1, color='blue', label='Expected saturation zone (10K-30K)')

ax.set_xlabel('Vocabulary Size (K)', fontsize=12)
ax.set_ylabel('Perplexity (lower = better)', fontsize=12)
ax.set_title('PAATRA Kill Gate: Vocabulary Size vs Perplexity', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Format x-axis as K
ax.set_xticks(k_values)
ax.set_xticklabels([f'{k//1000}K' for k in k_values])

# Add ratio annotations
for k, ppl in zip(k_values, ppls):
    ratio = ppl / baseline_ppl
    ax.annotate(f'{ratio:.1f}x', (k, ppl), textcoords='offset points',
               xytext=(0, 12), ha='center', fontsize=9, color='#e74c3c')

plt.tight_layout()
plt.savefig('paatra_step2_kill_gate.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: paatra_step2_kill_gate.png")

## 9. Verdict

In [ ]:
print("\n" + "="*60)
print("  KILL GATE VERDICT")
print("="*60)

# Find where perplexity stays within 2x of baseline
viable_threshold = 2.0  # PPL within 2x of baseline is "survivable"

for K in sorted([k for k in ppl_results if isinstance(k, int)]): # Filter out non-integer keys
    if K == "baseline":
        continue
    ratio = ppl_results[K] / baseline_ppl
    status = "PASS" if ratio < viable_threshold else "DEGRADED" if ratio < 5.0 else "FAIL"
    bar = "#" * min(int(ratio * 10), 50)
    print(f"  K={K:>6,}: {ratio:>5.2f}x baseline  [{status:>8s}]  {bar}")

print()
print("  PASS     = PPL < 2x baseline (model works, minor degradation)")
print("  DEGRADED = PPL 2-5x baseline (model struggles but produces language)")
print("  FAIL     = PPL > 5x baseline (model produces garbage)")
print()

# Find saturation point
for K in sorted([k for k in ppl_results if k != "baseline"]):
    if ppl_results[K] / baseline_ppl < viable_threshold:
        print(f"  >> Saturation point: ~{K//1000}K tokens")
        print(f"     Below this, the model still functions.")
        print(f"     The {tokenizer.vocab_size - K:,} pruned tokens were dead weight.")
        break
else:
    print("  >> No clear saturation point found. Model degrades at all subset sizes.")
    print("     Consider: (a) trying larger K values, (b) different frequency source, (c) different model.")

print()
print("IMPORTANT: This is zeroing-out without retraining.")
print("With proper cross-tokenizer KD (PAATRA Stage 3), results should be significantly better.")
print("The kill gate just tests whether the approach is fundamentally viable.")